In [1]:
!ls ../data/wikipedia/*/wiki*.jsonl

../data/wikipedia/Gemma4E4B/wikipedia_synthetic.jsonl
../data/wikipedia/Qwen36-27B/wikipedia_synthetic10.jsonl
../data/wikipedia/Qwen36-27B/wikipedia_synthetic11.jsonl
../data/wikipedia/Qwen36-27B/wikipedia_synthetic12.jsonl
../data/wikipedia/Qwen36-27B/wikipedia_synthetic13.jsonl
../data/wikipedia/Qwen36-27B/wikipedia_synthetic14.jsonl
../data/wikipedia/Qwen36-27B/wikipedia_synthetic15.jsonl
../data/wikipedia/Qwen36-27B/wikipedia_synthetic16.jsonl
../data/wikipedia/Qwen36-27B/wikipedia_synthetic1.jsonl
../data/wikipedia/Qwen36-27B/wikipedia_synthetic2.jsonl
../data/wikipedia/Qwen36-27B/wikipedia_synthetic3.jsonl
../data/wikipedia/Qwen36-27B/wikipedia_synthetic4.jsonl
../data/wikipedia/Qwen36-27B/wikipedia_synthetic5.jsonl
../data/wikipedia/Qwen36-27B/wikipedia_synthetic6.jsonl
../data/wikipedia/Qwen36-27B/wikipedia_synthetic7.jsonl
../data/wikipedia/Qwen36-27B/wikipedia_synthetic8.jsonl
../data/wikipedia/Qwen36-27B/wikipedia_synthetic9.jsonl


In [2]:
import json
import pandas as pd
from glob import glob
from pathlib import Path

PATH = "../data/wikipedia/*/wiki*.jsonl"
files = sorted(glob(PATH))

def load_json(file):
    model = Path(file).parent.name
    def get_data(raw):
        line = json.loads(raw)
        return {
            "text": line["text"],
            "labels": line.get("labels"),
            "not_labels": line.get("not_labels"),
            "model": model,
        }
    with open(file, "r") as f:
        data = [get_data(line) for line in f]
    return data

files_data = [load_json(file) for file in files]
df = pd.DataFrame([i for file_data in files_data for i in file_data])

In [3]:
df.sample(5)

,text,labels,not_labels,model
132710,Within merely five years of his inaugural Egyp...,"[professional_ascent, field_directorship, inst...","[formative_skill_origin, war_time_contribution...",Qwen36-27B
63455,PROCESS_HALT_SEQUENCE_INITIATED: Attempted res...,"[historical_context_mapping, status_transition...","[multi_modal_entity_resolution, information_in...",Gemma4E4B
134416,The system remained stable from 1960 through 2...,"[lifecycle_status, temporal_scope, deployment_...","[dependency_requirement, runtime_substitution,...",Qwen36-27B
114809,Discover peaceful countryside living with easy...,"[urban_proximity, nature_integration, lifestyl...","[generational_continuity, family_friendliness,...",Qwen36-27B
68712,Section 4.1: The establishment of successors t...,"[institutional_electoral_process, hereditary_o...","[military_intervention_authority, constitution...",Qwen36-27B


In [4]:
def merge_group(group):
    merged_labels = set().union(*group["labels"])
    merged_not_labels = set().union(*group["not_labels"])
    merged_not_labels -= merged_labels  # remove any intersection
    models = sorted(set(group["model"]))
    return pd.Series({
        "labels": sorted(merged_labels),
        "not_labels": sorted(merged_not_labels),
        "model": models if len(models) > 1 else models[0],
    })

df = (
    df.groupby("text", sort=False)
    .apply(merge_group, include_groups=False)
    .reset_index()
)
print(f"{len(df)} unique texts after merging")
df.sample(5)


183730 unique texts after merging


,text,labels,not_labels,model
37980,It is remarkable how a piece conceived in one ...,"[narrative_of_artistic_transition, recognition...","[acknowledgement_of_specific_achievement, cele...",Gemma4E4B
121928,The argumentative structure seems to prioritiz...,"[descriptive_over_analytical, interpretive_fra...","[evidentiary_gaps_identified, multi_period_com...",Qwen36-27B
90522,"In a quiet corner of England, a man transforms...","[heritage_reclamation, humanistic_medical_phil...","[career_pivot_narrative, institutional_adaptat...",Qwen36-27B
39009,Our proprietary algorithmic engine addresses a...,"[long_term_value_proposition, market_opportuni...","[competitive_landscape_analysis, disruptive_in...",Gemma4E4B
76116,Hold up do they play at Ahmed Fellak Stadium o...,"[casual_exclamation_tone, factual_correction_o...","[anecdotal_color_choice, casual_factual_inquir...",Qwen36-27B


In [5]:
import random

random.seed(42)

# A label is present around 3x in the data, 
# so we adjust the test ratio to get ~1% test rows after the strict split 
# that also excludes rows with test labels in "not_labels"
test_ratio = 0.01 / 3 

# All labels (positive + negative)
all_pos_labels = set(l for labs in df["labels"] for l in labs)
all_neg_labels = set(l for labs in df["not_labels"] for l in labs)

full_vocab = all_pos_labels | all_neg_labels

# Only choose test labels from positive labels
# (because they need to be ground truth in test)
candidate_test_labels = list(all_pos_labels)
random.shuffle(candidate_test_labels)

n_test_labels = int(len(all_pos_labels) * test_ratio)
test_labels = set(candidate_test_labels[:n_test_labels])

# Strict split:
# Test rows = rows containing at least one test label
test_mask = df["labels"].apply(
    lambda labs: bool(set(labs) & test_labels)
)

df_test = df[test_mask].copy()

# Train rows = rows containing NO test label anywhere
train_mask = df.apply(
    lambda row: (
        len(set(row["labels"]) & test_labels) == 0
        and len(set(row["not_labels"]) & test_labels) == 0
    ),
    axis=1
)

df_train = df[train_mask].copy()

# Reset index
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# Verify
train_seen = (
    set(l for labs in df_train["labels"] for l in labs)
    |
    set(l for labs in df_train["not_labels"] for l in labs)
)

test_seen = (
    set(l for labs in df_test["labels"] for l in labs)
    |
    set(l for labs in df_test["not_labels"] for l in labs)
)

intersection = test_labels & train_seen

print("Train rows:", len(df_train))
print("Test rows :", len(df_test))
print("Held-out test labels:", len(test_labels))
print("Intersection with train:", len(intersection))

Train rows: 179487
Test rows : 2094
Held-out test labels: 1004
Intersection with train: 0


In [6]:
import pandas as pd
from IPython.display import display

train_pos = set(lab for labs in df_train["labels"]     for lab in labs)
train_neg = set(lab for labs in df_train["not_labels"] for lab in labs)
test_pos  = set(lab for labs in df_test["labels"]      for lab in labs)
test_neg  = set(lab for labs in df_test["not_labels"]  for lab in labs)

total = len(test_pos) + len(test_neg)

# For a given target set, split by how the label was seen in train
def bucket(s):
    return {
        "seen in train as positive only":          len((s & train_pos) - train_neg),
        "seen in train as negative only":          len((s & train_neg) - train_pos),
        "seen in train as both pos and neg":       len(s & train_pos & train_neg),
        "never seen in train":                     len(s - train_pos - train_neg),
    }

tp = bucket(test_pos)   # labels used as ground-truth positives in test
tn = bucket(test_neg)   # labels used as hard negatives in test
index = list(tp.keys())

counts = pd.DataFrame({
    "used as positive in test": [tp[k] for k in index],
    "used as negative in test": [tn[k] for k in index],
    "total":                    [tp[k] + tn[k] for k in index],
}, index=index)
counts.index.name = "how the label was seen in train"

ratios = (counts / total * 100).round(1).astype(str) + "%"

print(f"Unique test positive labels: {len(test_pos)}   "
      f"Unique test negative labels: {len(test_neg)}   "
      f"Total: {total}\n")
print("── Counts ──")
display(counts)
print("── % of total test labels ──")
display(ratios)


Unique test positive labels: 6244   Unique test negative labels: 7080   Total: 13324

── Counts ──


,used as positive in test,used as negative in test,total
how the label was seen in train,,,
seen in train as positive only,588,1096,1684
seen in train as negative only,1084,851,1935
seen in train as both pos and neg,2659,4065,6724
never seen in train,1913,1068,2981


── % of total test labels ──


,used as positive in test,used as negative in test,total
how the label was seen in train,,,
seen in train as positive only,4.4%,8.2%,12.6%
seen in train as negative only,8.1%,6.4%,14.5%
seen in train as both pos and neg,20.0%,30.5%,50.5%
never seen in train,14.4%,8.0%,22.4%


In [7]:
import datasets

train_ds = datasets.Dataset.from_pandas(df_train)
test_ds = datasets.Dataset.from_pandas(df_test)

dataset = datasets.DatasetDict({
    "train": train_ds,
    "test": test_ds
})

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'not_labels', 'model'],
        num_rows: 179487
    })
    test: Dataset({
        features: ['text', 'labels', 'not_labels', 'model'],
        num_rows: 2094
    })
})

In [8]:
dataset.push_to_hub("alexneakameni/ZSHOT-HARDSET-v2", commit_description="Upload of ZSHOT-HARDSET-v2 with train/test split based on held-out labels.")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/alexneakameni/ZSHOT-HARDSET-v2/commit/e6976a2163181597a968eb3716f9b04c8d23dbc2', commit_message='Upload dataset', commit_description='Upload of ZSHOT-HARDSET-v2 with train/test split based on held-out labels.', oid='e6976a2163181597a968eb3716f9b04c8d23dbc2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alexneakameni/ZSHOT-HARDSET-v2', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alexneakameni/ZSHOT-HARDSET-v2'), pr_revision=None, pr_num=None)